# Step 4: Transformer

## 4.1 Ne Değişiyor?

Bu bölümde tek başlıklı attention yerine dört başlıklı
attention kullanacağız. Ayrıca modeli birden fazla
katmanla çalışabilecek şekilde düzenleyeceğiz.

Başlangıçta bir katman kullanacağız. Her katmanın
kendi parametreleri ve KV cache'i olacak.

In [1]:
import random
import math

random.seed(42)

with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

# Verideki benzersiz karakterler
uchars = sorted(set("".join(docs)))

# Karakter → token numarası
stoi = {ch: i for i, ch in enumerate(uchars)}

# Başlangıç ve bitiş için özel token
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Sözlük boyutu:", vocab_size)
print("BOS tokenı:", BOS)
print("Karakterler:", "".join(uchars))

İsim sayısı: 1100
Sözlük boyutu: 30


In [2]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data + other.data,
            (self, other),
            (1.0, 1.0),
        )

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data),
        )

    def __pow__(self, power):
        return Value(
            self.data ** power,
            (self,),
            (power * self.data ** (power - 1),),
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1.0 / self.data,),
        )

    def exp(self):
        result = math.exp(self.data)

        return Value(
            result,
            (self,),
            (result,),
        )

    def relu(self):
        return Value(
            max(0.0, self.data),
            (self,),
            (float(self.data > 0),),
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for child in node._children:
                    build_topo(child)

                topo.append(node)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            for child, local_grad in zip(
                node._children, node._local_grads
            ):
                child.grad += local_grad * node.grad

In [3]:
# y = x² + 2x
x = Value(3.0)
y = x * x + 2 * x

y.backward()

print("y değeri:", y.data)
print("x gradyanı:", x.grad)

assert math.isclose(y.data, 15.0)
assert math.isclose(x.grad, 8.0)

In [ ]:
def linear(x, w):
    return [
        sum(wi * xi for wi, xi in zip(row, x))
        for row in w
    ]


def softmax(logits):
    max_val = max(val.data for val in logits)

    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)

    return [e / total for e in exps]


def rmsnorm(x):
    mean_square = sum(xi * xi for xi in x) / len(x)
    scale = (mean_square + 1e-5) ** -0.5

    return [xi * scale for xi in x]

In [ ]:
n_embd = 16
block_size = 16

n_head = 4
n_layer = 1

# Embedding boyutu başlık sayısına tam bölünmeli.
assert n_embd % n_head == 0, (
    "n_embd, n_head sayısına tam bölünmeli."
)

head_dim = n_embd // n_head

print("Embedding boyutu:", n_embd)
print("Attention başlığı sayısı:", n_head)
print("Başlık başına boyut:", head_dim)
print("Katman sayısı:", n_layer)

## 4.2 Çok Başlı Attention

16 boyutlu Q, K ve V vektörlerini dört başlığa ayıracağız.
Her başlık dört boyut kullanacak ve kendi attention
ağırlıklarını hesaplayacak.

Başlıkların dört boyutlu çıktıları yan yana birleştirilerek
yeniden 16 boyutlu bir vektör oluşturulacak.

Başlıkların neye dikkat edeceği eğitim sırasında öğrenilecek.